# NVARC port -- ARC Prize 2026 (ARC-AGI-2)

Recreation of the ARC Prize 2025 winning solution by Ivan Sorokin and
Jean-Francois Puget (NVIDIA), re-pointed at the 2026 competition.
Generated by `submission/build_notebook.py` -- edit the sources in
`submission/src/`, not this notebook.

**Setup:** accelerator `L4x4`, internet **off**, attach the competition
dataset and the Kaggle model `sorokin/qwen3_2b_grids15_sft141`.


In [ ]:
import time, os
NOTEBOOK_START = time.time()
os.environ["ARC_MODEL_SLUG"] = "qwen3_2b_grids15_sft141"

# 'test'  -> the scored path (240 hidden tasks on a rerun, 4 smoke
#            tasks on a commit).
# 'eval'  -> the 120 public evaluation tasks, whose answers ship with
#            the competition. A commit run then prints a real accuracy
#            number and costs zero submissions. NVARC's published
#            baseline on this split is 25/120 for the 2B, 30/120 for 4B.
os.environ["ARC_TASK_SET"] = "test"
os.environ["ARC_TASK_LIMIT"] = "0"   # >0 samples the split
print("start", time.strftime("%H:%M:%S"), "| set", os.environ["ARC_TASK_SET"])

In [ ]:
import glob, os, subprocess, sys

# Locate the wheelhouse by content, not by path. Datasets mount at
# /kaggle/input/<slug>/ but notebook outputs land at
# /kaggle/input/notebooks/<owner>/<slug>/, so hunt for the wheels themselves.
found = glob.glob("/kaggle/input/**/unsloth-*.whl", recursive=True)
# Prefer the full dependency set over the --no-deps 'wheels_pinned' variant.
dirs = sorted({os.path.dirname(p) for p in found},
              key=lambda d: (d.endswith("wheels_pinned"), -len(os.listdir(d))))
assert dirs, (
    "No wheelhouse found under /kaggle/input. Internet is off in this "
    "competition, so pip cannot reach PyPI. Run submission/build_wheelhouse.py, "
    "push and run that notebook, then attach its output here via "
    "Add Input -> Notebook Output.\n"
    "Present: " + str(glob.glob("/kaggle/input/*") + glob.glob("/kaggle/input/*/*"))
)
WHEELS = dirs[0]
print("wheelhouse:", WHEELS, "->", len(os.listdir(WHEELS)), "files")

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tensorflow"],
               capture_output=True)

r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-index", "--find-links", WHEELS,
     "unsloth", "unsloth_zoo"],
    capture_output=True, text=True)
print(r.stdout[-3000:] or r.stderr[-3000:])
assert r.returncode == 0, "unsloth install failed -- see log above"

# flash-attn is optional: without it Unsloth falls back to SDPA, which is slower
# but correct. Only install if a matching prebuilt wheel is in the wheelhouse.
fa = glob.glob(os.path.join(WHEELS, "flash_attn-*.whl"))
if fa:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                        "--no-deps", fa[0]], capture_output=True, text=True)
    print("flash-attn:", "OK" if r.returncode == 0 else r.stderr[-1500:])
else:
    print("WARNING: no flash-attn wheel; falling back to SDPA (slower)")
    os.environ["NO_FLASH_ATTN"] = "1"

In [ ]:
PATCH_B64 = "QEAgLTI3Miw3ICsyNzcsNyBAQA0KIA0KICAgICAgICAgIyBNaXN0cmFsIE5lbW8gMTJiIGhhcyB3ZWlyZCBkaW1lbnNpb25zDQogICAgICAgICBpZiBhdHRlbnRpb25fc2l6ZSAhPSBoaWRkZW5fc2l6ZToNCi0gICAgICAgICAgICBzZWxmLnRlbXBfTyA9IHRvcmNoLmVtcHR5KCgxLCBic3osIGhpZGRlbl9zaXplKSwgZHR5cGUgPSBkdHlwZSwgZGV2aWNlID0gZGV2aWNlKQ0KKyAgICAgICAgICAgIHNlbGYudGVtcF9PID0gdG9yY2guZW1wdHkoKGJzeiwgMSwgaGlkZGVuX3NpemUpLCBkdHlwZSA9IGR0eXBlLCBkZXZpY2UgPSBkZXZpY2UpDQogICAgICAgICBlbHNlOg0KICAgICAgICAgICAgIHNlbGYudGVtcF9PID0gc2VsZi50ZW1wX1FBWzFdWzosOiw6aGlkZGVuX3NpemVdDQogICAgICAgICBwYXNzDQpAQCAtMzMzLDUyICszMzgsMTIgQEANCiAgICAgS24gPSBzZWxmLnBhZ2VkX2F0dGVudGlvbl9LWzprdl9zZXFfbGVuXS5wZXJtdXRlKDEsIDIsIDAsIDMpDQogICAgIFZuID0gc2VsZi5wYWdlZF9hdHRlbnRpb25fVls6a3Zfc2VxX2xlbl0ucGVybXV0ZSgxLCAyLCAwLCAzKQ0KIA0KLSAgICAjIEhhbmRsZSBzbGlkaW5nIHdpbmRvd3MNCi0gICAgc2xpZGluZ193aW5kb3cgPSBnZXRhdHRyKHNlbGYuY29uZmlnLCAic2xpZGluZ193aW5kb3ciLCBOb25lKQ0KLSAgICBpZiBzbGlkaW5nX3dpbmRvdyBpcyBub3QgTm9uZSBhbmQga3Zfc2VxX2xlbiA+IHNsaWRpbmdfd2luZG93Og0KLSAgICAgICAgIyBGcm9tIGh0dHBzOi8vZ2l0aHViLmNvbS9odWdnaW5nZmFjZS90cmFuc2Zvcm1lcnMvYmxvYi9tYWluL3NyYy90cmFuc2Zvcm1lcnMvbW9kZWxzL21pc3RyYWwvbW9kZWxpbmdfbWlzdHJhbC5weSNMMTkzDQotICAgICAgICBzbGljaW5nX3Rva2VucyA9IDEgLSBzbGlkaW5nX3dpbmRvdw0KLSAgICAgICAgS25uID0gS25bOiwgOiwgc2xpY2luZ190b2tlbnM6LCA6XSMuY29udGlndW91cygpDQotICAgICAgICBWbm4gPSBWbls6LCA6LCBzbGljaW5nX3Rva2VuczosIDpdIy5jb250aWd1b3VzKCkNCi0gICAgZWxzZToNCi0gICAgICAgIEtubiwgVm5uID0gS24sIFZuDQotICAgIHBhc3MNCisgICAgUW5uID0gUW4udHJhbnNwb3NlKDEsIDIpDQorICAgIEtubiA9IEtuLnRyYW5zcG9zZSgxLCAyKQ0KKyAgICBWbm4gPSBWbi50cmFuc3Bvc2UoMSwgMikNCiANCi0gICAgIyB3aGVuIHFsZW49PXZsZW4gYW5kIGF0dG5fbWFzayBpcyBOb25lLCB3ZSBzaG91bGQgdXNlIGNhdXNhbCBhdHRlbnRpb24NCi0gICAgUV9sZW4gPSBRbi5zaGFwZVstMl0NCi0gICAgS19sZW4gPSBLbm4uc2hhcGVbLTJdDQotICAgIGlmIGF0dGVudGlvbl9tYXNrIGlzIE5vbmUgYW5kIFFfbGVuID09IEtfbGVuOg0KLSAgICAgICAgaXNfY2F1c2FsID0gVHJ1ZQ0KLSAgICBlbHNlOg0KLSAgICAgICAgaXNfY2F1c2FsID0gRmFsc2UNCisgICAgQSA9IGZsYXNoX2F0dG5fZnVuYyhRbm4sIEtubiwgVm5uKQ0KIA0KLSAgICAjIEdyb3VwZWQgcXVlcnkgYXR0ZW50aW9uDQotICAgIF8sIF8sIGNhY2hlZF9sZW4sIF8gPSBLbm4uc2hhcGUNCi0gICAgaWYgYnN6ID09IDEgb3Igbm90IFNEUEFfSEFTX0dRQSBhbmQgbl9ncm91cHMgIT0gMToNCi0gICAgICAgIEtubiA9IEtubls6LCA6LCBOb25lLCA6LCA6XS5leHBhbmQoYnN6LCBuX2t2X2hlYWRzLCBuX2dyb3VwcywgY2FjaGVkX2xlbiwgaGVhZF9kaW0pDQotICAgICAgICBWbm4gPSBWbm5bOiwgOiwgTm9uZSwgOiwgOl0uZXhwYW5kKGJzeiwgbl9rdl9oZWFkcywgbl9ncm91cHMsIGNhY2hlZF9sZW4sIGhlYWRfZGltKQ0KLSAgICAgICAgS25uID0gS25uLnJlc2hhcGUoYnN6LCBuX2hlYWRzLCBjYWNoZWRfbGVuLCBoZWFkX2RpbSkNCi0gICAgICAgIFZubiA9IFZubi5yZXNoYXBlKGJzeiwgbl9oZWFkcywgY2FjaGVkX2xlbiwgaGVhZF9kaW0pDQotICAgIHBhc3MNCi0gICAgIyBlbHNlOg0KLSAgICAjICAgICBLbm4sIFZubiA9IEtubiwgVm5uDQotICAgICMgcGFzcw0KLQ0KLSAgICAjIEF0dGVudGlvbg0KLSAgICBpZiBic3ogPT0gMToNCi0gICAgICAgIFFuICo9IHNlbGYuc2NhbGFyICMgU2VlIGh0dHBzOi8vZ2l0aHViLmNvbS9nZ2VyZ2Fub3YvbGxhbWEuY3BwL2lzc3Vlcy83ODA1I2lzc3VlY29tbWVudC0yMTUzMzQ5OTYzDQotICAgICAgICAjIEl0IHNlZW1zIGxpa2UgZG9pbmcgKFEgKiBzY2FsYXIpIEAgSyBpcyBiZXR0ZXIgdGhhbiAoUSBAIEspICogc2NhbGFyIHRvIHN0b3Agb3ZlcmZsb3dzDQotICAgICAgICBBID0gdG9yY2hfbWF0bXVsKFFuLCBLbm4udHJhbnNwb3NlKDIsIDMpLCBvdXQgPSBzZWxmLmF0dGVudGlvbls6LDosOiw6Y2FjaGVkX2xlbl0pDQotICAgICAgICAjIGlmIGF0dGVudGlvbl9tYXNrIGlzIG5vdCBOb25lOiBBICs9IGF0dGVudGlvbl9tYXNrICMgTXVzdCBhZGQgYXR0ZW50aW9uX21hc2sgZm9yIGJhdGNoZWQNCi0gICAgICAgIEFbOl0gPSB0b3JjaF9ubl9mdW5jdGlvbmFsX3NvZnRtYXgoQSwgZGltID0gLTEsIGR0eXBlID0gdG9yY2guZmxvYXQzMikjLnRvKEEuZHR5cGUpDQotICAgICAgICBBID0gdG9yY2hfbWF0bXVsKEEsIFZubiwgb3V0ID0gUW4pDQotICAgIGVsc2U6DQotICAgICAgICBpZiBTRFBBX0hBU19HUUE6DQotICAgICAgICAgICAgQSA9IHNjYWxlZF9kb3RfcHJvZHVjdF9hdHRlbnRpb24oUW4sIEtubiwgVm5uLCBhdHRuX21hc2sgPSBhdHRlbnRpb25fbWFzaywgaXNfY2F1c2FsID0gaXNfY2F1c2FsLCBlbmFibGVfZ3FhID0gVHJ1ZSkNCi0gICAgICAgIGVsc2U6DQotICAgICAgICAgICAgQSA9IHNjYWxlZF9kb3RfcHJvZHVjdF9hdHRlbnRpb24oUW4sIEtubiwgVm5uLCBhdHRuX21hc2sgPSBhdHRlbnRpb25fbWFzaywgaXNfY2F1c2FsID0gaXNfY2F1c2FsKQ0KLSAgICBwYXNzDQotICAgIEEgPSBBLnRyYW5zcG9zZSgxLCAyKQ0KICAgICBBID0gQS5yZXNoYXBlKGJzeiwgMSwgYXR0ZW50aW9uX3NpemUpDQogICAgIEEgPSBmYXN0X2xpbmVhcl9mb3J3YXJkKHNlbGYub19wcm9qLCBBLCBvdXQgPSBzZWxmLnRlbXBfTykNCiAgICAgcmV0dXJuIEEsIChLbiwgVm4pDQo="

In [ ]:
import base64, os, subprocess, sys

# The patch rewrites Unsloth's inference path to call flash_attn_func, so it is
# only valid when flash-attn is installed. Without it, leave Unsloth on SDPA:
# batched decoding still works, just slower.
try:
    import flash_attn
    HAVE_FA = True
    print("flash_attn", flash_attn.__version__)
except ImportError:
    HAVE_FA = False
    print("flash_attn absent -- skipping patch, Unsloth will use SDPA")

if HAVE_FA:
    with open("qwen3.patch", "wb") as f:
        f.write(base64.b64decode(PATCH_B64))

    import unsloth.models
    target = os.path.join(os.path.dirname(unsloth.models.__file__), "qwen3.py")
    print("patch target:", target)

    r = subprocess.run(["patch", "--binary", "--forward", target, "qwen3.patch"],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)

    src = open(target).read()
    assert "flash_attn_func(Qnn, Knn, Vnn)" in src, (
        "qwen3.py is NOT patched. Batched DFS decoding wants Unsloth's inference "
        "path on flash_attn_func with bsz>1. Check that unsloth==2025.9.7 "
        "installed and that the patch hunks still apply to this version."
    )
    print("OK: unsloth qwen3 inference path patched for batched decoding")

In [ ]:
%%writefile arc_config.py
"""Central configuration for the ARC Prize 2026 port of the NVARC solution.

Everything that differs between NVARC's original ARC-AGI-1 notebook and this
2026 submission lives here, so `arc_loader.py` and `arc_decoder.py` stay
byte-identical to the originals and `arc_solver.py` keeps a small diff.
"""

import os
import glob
import time


# --- Which model to run ------------------------------------------------------
# Attach the Kaggle model, then set MODEL_SLUG to its directory name.
#   sorokin/qwen3_2b_grids15_sft141 -> 22.22% public LB in 6h10min (paper Table 2)
#   sorokin/qwen3_4b_grids15_sft139 -> 29.72% public LB in 12h     (paper Table 2)
MODEL_SLUG = os.environ.get("ARC_MODEL_SLUG", "qwen3_2b_grids15_sft141")


# --- Paths -------------------------------------------------------------------
COMPETITION = "arc-prize-2026-arc-agi-2"
INPUT_DIR = os.environ.get("ARC_INPUT_DIR", "/kaggle/input")
WORK_DIR = os.environ.get("ARC_WORK_DIR", "/kaggle/working")

TEST_FILE = "arc-agi_test_challenges.json"


def find_data_dir():
    """Locate the competition data by filename rather than by mount path.

    Kaggle's mount layout is not one thing: datasets land at
    /kaggle/input/<slug>/, notebook outputs at
    /kaggle/input/notebooks/<owner>/<slug>/, and the competition directory name
    does not always equal the competition slug. Search for the file instead.
    """
    if os.environ.get("ARC_DATA_DIR"):
        return os.environ["ARC_DATA_DIR"]

    direct = os.path.join(INPUT_DIR, COMPETITION)
    if os.path.exists(os.path.join(direct, TEST_FILE)):
        return direct

    hits = glob.glob(os.path.join(INPUT_DIR, "**", TEST_FILE), recursive=True)
    if hits:
        return os.path.dirname(sorted(hits, key=len)[0])

    raise FileNotFoundError(
        f"Could not find {TEST_FILE} anywhere under {INPUT_DIR}. Attach the "
        f"'{COMPETITION}' competition to this notebook.\n"
        f"Present: {sorted(glob.glob(os.path.join(INPUT_DIR, '*')))}"
    )


DATA_DIR = find_data_dir() if os.path.isdir(INPUT_DIR) else os.path.join(
    INPUT_DIR, COMPETITION)

TEST_CHALLENGES = os.path.join(DATA_DIR, TEST_FILE)
EVAL_CHALLENGES = os.path.join(DATA_DIR, "arc-agi_evaluation_challenges.json")
EVAL_SOLUTIONS = os.path.join(DATA_DIR, "arc-agi_evaluation_solutions.json")


# --- Which task set to solve -------------------------------------------------
# "test" is the scored path. "eval" solves the 120 public evaluation tasks,
# whose answers ship with the competition, so a commit run yields a real
# accuracy number without spending one of the 1-per-day submissions.
#
# Comparable published baseline: NVARC's paper (Table 2) reports 25/120 on this
# same split for the 2B checkpoint, and 30/120 for the 4B. Both were measured
# with the same contamination we have -- 55% of their training data was seeded
# from these very puzzles' descriptions -- so the comparison is apples to
# apples even though the absolute number reads high.
TASK_SET = os.environ.get("ARC_TASK_SET", "test")
assert TASK_SET in ("test", "eval"), TASK_SET

CHALLENGES = TEST_CHALLENGES if TASK_SET == "test" else EVAL_CHALLENGES
SOLUTIONS = None if TASK_SET == "test" else EVAL_SOLUTIONS

# 0 = every task. Set lower to sample the split for a faster signal.
TASK_LIMIT = int(os.environ.get("ARC_TASK_LIMIT", 0))

# NVARC wrote these to '../inference_outputs' and '../worker{rank}', which on
# Kaggle resolve to /kaggle/ -- not writable. Keep them under /kaggle/working.
OUTPUT_DIR = os.path.join(WORK_DIR, "inference_outputs")
WORKER_FLAG_DIR = os.path.join(WORK_DIR, "worker_flags")
SUBMISSION_PATH = os.path.join(WORK_DIR, "submission.json")


def model_path():
    """Resolve an attached Kaggle model to its weights directory.

    Verified layout (2026-08):
        /kaggle/input/models/<owner>/<slug>/<framework>/<variation>/<version>/
    The owner prefix and version both vary, so search for config.json under any
    directory whose path contains the slug rather than assuming a depth.
    """
    if os.environ.get("ARC_MODEL_DIR"):
        return os.environ["ARC_MODEL_DIR"]

    hits = [p for p in glob.glob(os.path.join(INPUT_DIR, "**", "config.json"),
                                 recursive=True)
            if MODEL_SLUG in p]
    if hits:
        # Shortest path wins: avoids nested subdirs like ./checkpoint/config.json
        return os.path.dirname(sorted(hits, key=len)[0])

    raise FileNotFoundError(
        f"No config.json for '{MODEL_SLUG}' under {INPUT_DIR}. Attach the Kaggle "
        f"model 'sorokin/{MODEL_SLUG}' to this notebook, or set ARC_MODEL_DIR.\n"
        f"Present: {sorted(glob.glob(os.path.join(INPUT_DIR, '*')))}"
    )


# --- Run mode ----------------------------------------------------------------
def is_rerun():
    """True during the scored rerun against the 240 hidden tasks.

    On the interactive/commit run Kaggle supplies a placeholder
    arc-agi_test_challenges.json. In 2026 that placeholder is 240 *training*
    tasks (verified: 240/240 overlap with arc-agi_training_challenges.json,
    0/240 with the evaluation set), so any score computed from it is
    meaningless. Solve only a handful there to keep commits fast.
    """
    return bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))


# Tasks solved during a non-rerun commit, purely to exercise the pipeline.
SMOKE_TEST_KEYS = ["00576224", "007bbfb7", "009d5c81", "00d62c1b"]


# --- Time budget -------------------------------------------------------------
# Kaggle hard-kills at 12h and obfuscates reported runtime by up to ~10 min, so
# leave more headroom than NVARC's 600s. Anchored at import time.
TOTAL_BUDGET_S = float(os.environ.get("ARC_TOTAL_BUDGET_S", 12 * 3600))
SAFETY_MARGIN_S = float(os.environ.get("ARC_SAFETY_MARGIN_S", 1200))

# Per-puzzle wall clock before we stop decoding and move on (NVARC used 1200).
PUZZLE_BUDGET_S = float(os.environ.get("ARC_PUZZLE_BUDGET_S", 1200))
# Per-DFS-call cap (NVARC used 540).
DFS_BUDGET_S = float(os.environ.get("ARC_DFS_BUDGET_S", 540))

NUM_WORKERS = int(os.environ.get("ARC_NUM_WORKERS", 4))  # one per L4

_START = time.time()


def deadline():
    return _START + TOTAL_BUDGET_S - SAFETY_MARGIN_S


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_mask.py
"""Completion-only label masking.

Kept in its own module, free of unsloth/torch imports, so the masking rule can
be unit tested offline. `arc_solver` calls it from the data collator.
"""

import numpy as np

IGNORE_INDEX = -100


def completion_labels(ids, im_start_id, eos_id, newline_id,
                      ignore_index=IGNORE_INDEX):
    """Supervise only the assistant turns of a chat-formatted sequence.

    Turns alternate user, assistant, user, assistant, ... Each turn looks like

        <|im_start|> [role] \\n  <content...>  <|im_end|>

    and we train on the content plus the closing <|im_end|>, so the model also
    learns where to stop. The role word is optional: on some checkpoints the
    tokenizer drops it, so we locate content by scanning to the newline that
    terminates the header rather than by assuming a fixed offset.

    Returns an int64 array the same length as `ids`.
    """
    ids = np.asarray(ids)
    labels = np.full(ids.shape, ignore_index, dtype=np.int64)

    starts = np.where(ids == im_start_id)[0].tolist()
    ends = np.where(ids == eos_id)[0].tolist()

    for turn, (start, end) in enumerate(zip(starts, ends)):
        if turn % 2 != 1:           # even turns are prompts
            continue
        nl = start + 1
        while nl < end and int(ids[nl]) != newline_id:
            nl += 1
        content = nl + 1
        stop = end + 1              # inclusive of <|im_end|>
        if content < stop:
            labels[content:stop] = ids[content:stop]

    return labels


In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter
from arc_mask import completion_labels

import arc_config

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15
IM_START_ID = 14
NEWLINE_ID = 10


def init_vocab(tokenizer, rank=0):
    """Resolve the special token ids from the tokenizer actually in use.

    NVARC hardcoded these for the 4B checkpoint's 16-token vocabulary. The 2B
    checkpoint is cut from a different base model, so the ids need not match --
    and if `user`/`assistant` resolve elsewhere, the completion-only collator
    finds no turn boundaries, masks every label, and the loss becomes NaN over
    zero targets. Derive them instead, and verify against a real encoding.
    """
    global ARC_TOKENS, USER_TOKEN_ID, ASSISTANT_TOKEN_ID, PAD_ID, EOS_ID
    global IM_START_ID, NEWLINE_ID

    def tid(tok, default=None):
        try:
            i = tokenizer.convert_tokens_to_ids(tok)
        except Exception:
            return default
        return default if i is None or i < 0 else int(i)

    USER_TOKEN_ID = tid("user", USER_TOKEN_ID)
    ASSISTANT_TOKEN_ID = tid("assistant", ASSISTANT_TOKEN_ID)
    EOS_ID = tid("<|im_end|>", EOS_ID)
    IM_START_ID = tid("<|im_start|>", IM_START_ID)
    NEWLINE_ID = tid("Ċ", NEWLINE_ID)
    PAD_ID = (int(tokenizer.pad_token_id)
              if getattr(tokenizer, "pad_token_id", None) is not None
              else tid("<|endoftext|>", PAD_ID))

    digits = [tid(str(d)) for d in range(10)]
    newline = tid("Ċ")
    ARC_TOKENS = [t for t in digits + [newline, EOS_ID] if t is not None]

    print(f"[Rank {rank}] vocab: user={USER_TOKEN_ID} assistant={ASSISTANT_TOKEN_ID} "
          f"eos={EOS_ID} pad={PAD_ID} arc_tokens={ARC_TOKENS}")

    # Verify against text in the exact shape the formatter emits.
    probe = "<|im_start|>user\n1<|im_end|><|im_start|>assistant\n2<|im_end|>"
    try:
        ids = tokenizer.encode(probe)
        ok = USER_TOKEN_ID in ids and ASSISTANT_TOKEN_ID in ids and EOS_ID in ids
        print(f"[Rank {rank}] vocab probe ids={ids} turn_markers_found={ok}")
        if not ok:
            print(f"[Rank {rank}] !!! turn markers absent from encoded text -- "
                  f"the collator cannot find turn boundaries and every label "
                  f"will be masked (NaN loss)")
    except Exception as e:
        print(f"[Rank {rank}] vocab probe failed: {type(e).__name__}: {e}")


def diagnose_tokenizer(tokenizer, rank=0):
    """Two checks that would otherwise fail silently and cost a submission.

    1. Compare unsloth's tokenizer against a plain AutoTokenizer load of the same
       directory. The checkpoint's tokenizer.json is byte-identical to the copy
       in our repo, which *does* emit the `user` / `assistant` ids when driven
       directly -- so if the plain load works and unsloth's does not, the damage
       is being done while loading, not by the file.
    2. Round-trip a grid: text -> ids -> text. Decoding is how candidate answers
       become grids again, and if newlines do not survive, every answer parses as
       a single row -- wrong shape, zero score, no error anywhere.
    """
    if rank != 0:
        return

    probe = "<|im_start|>user\n1<|im_end|><|im_start|>assistant\n2<|im_end|>"
    try:
        from transformers import AutoTokenizer
        plain = AutoTokenizer.from_pretrained(arc_config.model_path(),
                                              local_files_only=True)
        theirs, ours = plain.encode(probe), tokenizer.encode(probe)
        print(f"[Rank {rank}] AutoTokenizer : {theirs}")
        print(f"[Rank {rank}] unsloth       : {ours}")
        if theirs != ours:
            print(f"[Rank {rank}] !!! unsloth altered the tokenizer; the plain "
                  f"load is the correct behaviour")
    except Exception as e:
        print(f"[Rank {rank}] tokenizer comparison failed: {type(e).__name__}: {e}")

    grid_text = "123\n456\n789"
    try:
        ids = tokenizer.encode(grid_text)
        back = tokenizer.decode(ids)
        rows = [r for r in back.strip().split("\n") if r]
        ok = back.strip() == grid_text and len(rows) == 3
        print(f"[Rank {rank}] grid round-trip ids={ids} -> {back!r} rows={len(rows)} ok={ok}")
        if not ok:
            print(f"[Rank {rank}] !!! grids do not survive decode -- answers will "
                  f"parse with the wrong shape and score zero")
    except Exception as e:
        print(f"[Rank {rank}] grid round-trip failed: {type(e).__name__}: {e}")


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        # Supervise only the assistant turns, i.e. the answer grids.
        #
        # NVARC located turn starts by the `user` / `assistant` token ids. On this
        # checkpoint those words are dropped during encoding -- the ids resolve
        # fine via convert_tokens_to_ids, but tokenizer.encode() never emits them
        # -- so no boundary was ever found, every label stayed -100, and the loss
        # became NaN over zero targets. Anchor on <|im_start|> / <|im_end|>, which
        # do survive, and find the newline that terminates the role header. That
        # works whether or not the role word is present.
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            ids = batch["input_ids"][i]
            labels = completion_labels(
                ids.cpu().numpy(), IM_START_ID, EOS_ID, NEWLINE_ID)
            batch["labels"][i] = torch.as_tensor(
                labels, dtype=batch["labels"].dtype, device=batch["labels"].device)
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < arc_config.DFS_BUDGET_S and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=arc_config.model_path(),
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    init_vocab(tokenizer, rank)
    diagnose_tokenizer(tokenizer, rank)

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    # NVARC cast every fp32 param to bf16 to save VRAM. Unsloth now deliberately
    # keeps the embedding adapters in fp32 ("Training embed_tokens in mixed
    # precision"), so stomping those to bf16 fights its own scheme. We have room
    # to spare on L4 (peak 8.6 of 22 GB), so leave them alone.
    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            if "embed_tokens" in name or "lm_head" in name:
                continue
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    arc_test_set = ArcDataset.from_file(arc_config.CHALLENGES)

    dir_outputs = arc_config.OUTPUT_DIR
    os.makedirs(dir_outputs, exist_ok=True)

    worker_start = time.time()
    num_done = 0

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break

        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        # One-off check that the completion-only collator still masks correctly.
        # If transformers changed DataCollatorForLanguageModeling under us, every
        # label could come back -100, and a loss over zero targets is NaN -- which
        # looks identical to an exploding-gradient NaN in the training stats.
        if num_done == 0:
            try:
                probe = [{"input_ids": tokenizer.encode(s["text"])}
                         for s in train_ds.as_list(formatter)[:2]]
                b = collator(probe)
                n_lab = int((b["labels"] != -100).sum())
                n_tok = int(b["labels"].numel())
                print(f"[Rank {rank}] collator check: {n_lab}/{n_tok} tokens "
                      f"supervised ({100 * n_lab / max(n_tok, 1):.1f}%)")
                if n_lab == 0:
                    print(f"[Rank {rank}] !!! collator masks EVERYTHING -- "
                          f"loss will be NaN regardless of the model")
            except Exception as e:
                print(f"[Rank {rank}] collator check failed: {type(e).__name__}: {e}")

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        # A NaN loss means test-time fine-tuning learned nothing for this puzzle.
        # The run still completes and still emits candidates, so this would be
        # invisible without saying it out loud.
        loss = stats.metrics.get("train_loss") if hasattr(stats, "metrics") else None
        if loss is None or loss != loss:
            print(f"[Rank {rank}] !!! NaN training loss on {key} -- TTFT is not "
                  f"adapting the model; predictions are effectively untrained")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > arc_config.PUZZLE_BUDGET_S or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")

        spend_time = time.time() - start_time
        num_done += 1

        # Pace check. 240 tasks over NUM_WORKERS must clear the deadline; at
        # NVARC's 1200s ceiling they would not, so watch the running average.
        avg = (time.time() - worker_start) / num_done
        affordable = max(0, int((end_time - time.time()) / avg)) if avg > 0 else 0
        print(
            f"[Rank {rank}] finished {key} in {spend_time:.1f}s "
            f"| done={num_done} avg={avg:.1f}s "
            f"| {affordable} more fit before deadline"
        )

In [ ]:
%%writefile make_submission.py
"""Aggregate per-puzzle DFS candidates into submission.json.

This is NVARC's notebook cell 9, split out so it can run standalone and so a
valid submission always exists on disk even if the solver never finishes.
"""

import os
import json
import argparse

import numpy as np

import arc_config
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_kgmon, score_full_probmul_3


SELECTORS = {
    # NVARC's post-deadline formula (paper 3.4): DFS hit count + mean augmented
    # log-prob. This is what the vendored notebook defaults to, and what the
    # 29.72% run used.
    "kgmon": score_kgmon,
    # The original ARChitects product-of-experts score.
    "probmul": score_full_probmul_3,
}


def write_fallback(challenges_path=None, out_path=None):
    """Emit a format-valid submission covering every task, before any GPU work.

    Kaggle scores a missing or malformed submission.json as a hard failure, so
    write the all-[[0]] skeleton first and overwrite it later with real answers.
    """
    challenges_path = challenges_path or arc_config.CHALLENGES
    out_path = out_path or arc_config.SUBMISSION_PATH

    data = ArcDataset.from_file(challenges_path)
    submission = data.get_submission()
    with open(out_path, "w") as f:
        json.dump(submission, f)
    print(f"*** Wrote fallback submission for {len(submission)} tasks -> {out_path}")
    return submission


def validate_format(submission, challenges):
    """Fail loudly on the ways a submission silently scores zero."""
    problems = []

    missing = set(challenges) - set(submission)
    extra = set(submission) - set(challenges)
    if missing:
        problems.append(f"{len(missing)} task ids missing, e.g. {sorted(missing)[:3]}")
    if extra:
        problems.append(f"{len(extra)} unexpected task ids, e.g. {sorted(extra)[:3]}")

    for key, task in challenges.items():
        entry = submission.get(key)
        if entry is None:
            continue
        if not isinstance(entry, list) or len(entry) != len(task["test"]):
            problems.append(
                f"{key}: expected {len(task['test'])} outputs, got "
                f"{len(entry) if isinstance(entry, list) else type(entry).__name__}"
            )
            continue
        for i, attempts in enumerate(entry):
            for name in ("attempt_1", "attempt_2"):
                grid = attempts.get(name)
                if grid is None:
                    problems.append(f"{key}[{i}]: {name} absent")
                    continue
                if not isinstance(grid, list) or not grid or not isinstance(grid[0], list):
                    problems.append(f"{key}[{i}].{name}: not a 2D list")
                    continue
                widths = {len(r) for r in grid}
                if len(widths) != 1 or 0 in widths:
                    problems.append(f"{key}[{i}].{name}: ragged or empty rows")
                elif len(grid) > 30 or max(widths) > 30:
                    problems.append(f"{key}[{i}].{name}: exceeds 30x30")
                elif not all(isinstance(c, int) and 0 <= c <= 9 for r in grid for c in r):
                    problems.append(f"{key}[{i}].{name}: cells outside 0-9 int")

    return problems


def build(selector="kgmon", challenges_path=None, solutions_path=None, out_path=None):
    challenges_path = challenges_path or arc_config.CHALLENGES
    solutions_path = solutions_path or arc_config.SOLUTIONS
    out_path = out_path or arc_config.SUBMISSION_PATH

    data = ArcDataset.from_file(challenges_path)
    if solutions_path:
        data = data.load_replies(solutions_path)

    decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
    decoder.load_decoded_results(arc_config.OUTPUT_DIR)
    print(f"*** Loaded candidates for {len(decoder.decoded_results)} test outputs")

    submission = data.get_submission(decoder.run_selection_algo(SELECTORS[selector]))

    with open(challenges_path) as f:
        challenges = json.load(f)
    problems = validate_format(submission, challenges)
    if problems:
        print(f"!!! {len(problems)} FORMAT PROBLEMS")
        for p in problems[:20]:
            print("   ", p)
        raise SystemExit(1)
    print(f"*** Format OK: {len(submission)} tasks, "
          f"{sum(len(v) for v in submission.values())} outputs, 2 attempts each")

    with open(out_path, "w") as f:
        json.dump(submission, f)
    print(f"*** Wrote {out_path}")

    if solutions_path:
        score = data.validate_submission(submission)
        n = len(data.keys)
        print(f"*** Local score: {score:.2f}/{n} = {100 * score / n:.2f}%")

    return submission


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--selector", default="kgmon", choices=sorted(SELECTORS))
    p.add_argument("--challenges", default=None)
    p.add_argument("--solutions", default=None, help="score locally if given")
    p.add_argument("--out", default=None)
    p.add_argument("--fallback-only", action="store_true")
    a = p.parse_args()

    if a.fallback_only:
        write_fallback(a.challenges, a.out)
    else:
        build(a.selector, a.challenges, a.solutions, a.out)


In [ ]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp

import arc_config


def local_worker(rank, queue, end_time):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue: ranks must import serially, not concurrently.
    flag = os.path.join(arc_config.WORKER_FLAG_DIR, f"worker{rank}")
    prev = os.path.join(arc_config.WORKER_FLAG_DIR, f"worker{rank-1}")
    if rank > 0:
        while not os.path.exists(prev):
            time.sleep(5)

    from arc_solver import worker

    with open(flag, "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!", flush=True)

    worker(rank, queue, end_time)

    print(f"[Rank {rank}] done!", flush=True)


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    end_time = args.end_time or arc_config.deadline()

    rerun_mode = arc_config.is_rerun()

    with open(arc_config.CHALLENGES, "r") as f:
        data = json.load(f)

    os.makedirs(arc_config.WORKER_FLAG_DIR, exist_ok=True)
    os.makedirs(arc_config.OUTPUT_DIR, exist_ok=True)

    keys = sorted(data.keys())
    if arc_config.TASK_SET == "eval":
        # Public evaluation split: answers ship with the competition, so solving
        # it in a commit run gives a real accuracy number for zero submissions.
        if arc_config.TASK_LIMIT:
            keys = keys[:arc_config.TASK_LIMIT]
    elif not rerun_mode:
        # Commit run on the scored path: the shipped test file is a placeholder
        # of training tasks, so solving all of it is meaningless and slow. Solve
        # a few to prove the pipeline; make_submission.py fills in the rest.
        keys = [k for k in keys if k in arc_config.SMOKE_TEST_KEYS] or keys[:4]

    print(
        f"*** set={arc_config.TASK_SET} rerun={rerun_mode} tasks={len(keys)}/{len(data)} "
        f"workers={arc_config.NUM_WORKERS} "
        f"budget={(end_time - time.time()) / 3600:.2f}h",
        flush=True,
    )

    queue = mp.Manager().Queue()
    for key in keys:
        queue.put(key)
    for _ in range(arc_config.NUM_WORKERS):
        queue.put(None)

    mp.spawn(local_worker, args=(queue, end_time), nprocs=arc_config.NUM_WORKERS)


In [ ]:
# Write a format-valid submission before any GPU work, so a crash or a
# 12h timeout still leaves a scoreable file on disk.
import sys; sys.path.insert(0, '.')
import arc_config
from make_submission import write_fallback
write_fallback()

In [ ]:
# Budget is anchored at notebook start, so pip install time counts.
end_time = NOTEBOOK_START + arc_config.TOTAL_BUDGET_S - arc_config.SAFETY_MARGIN_S
print(f"solver budget: {(end_time - time.time()) / 3600:.2f}h")

In [ ]:
# A bare `!python` cannot fail a papermill cell, so a crashed solver would
# look like a green run that quietly submits all-[[0]]. Capture the code.
import subprocess, os
env = dict(os.environ, UNSLOTH_DISABLE_STATISTICS='1',
           TRITON_PTXAS_PATH='/usr/local/cuda/bin/ptxas',
           OMP_NUM_THREADS='12')
proc = subprocess.run(['python', 'starter.py', '--end-time', str(end_time)],
                      env=env)
SOLVER_RC = proc.returncode
print('solver exit code:', SOLVER_RC)

In [ ]:
# Rank candidates and overwrite the fallback. Raises if the format is bad.
!python make_submission.py --selector kgmon
print(f"total elapsed: {(time.time() - NOTEBOOK_START) / 3600:.2f}h")

In [ ]:
import json
sub = json.load(open(arc_config.SUBMISSION_PATH))
n_out = sum(len(v) for v in sub.values())
solved = sum(1 for v in sub.values()
             for a in v if a['attempt_1'] != [[0]])
print(f'{len(sub)} tasks, {n_out} outputs, {solved} with a real prediction')
print('solver exit code:', SOLVER_RC)

if solved == 0:
    msg = ('ZERO real predictions -- the solver produced nothing. This '
           'submission would score 0.00. Check the solver traceback above; '
           'the usual cause is no GPU (needs L4x4) or a bad model path.')
    if arc_config.is_rerun():
        # Scored rerun: keep the format-valid fallback rather than erroring
        # out with no submission at all, but make the failure unmissable.
        print('!!! ' + msg)
    else:
        # Commit run: fail loudly so this never gets promoted to a submission.
        raise AssertionError(msg)
else:
    print('OK: solver produced real predictions')